In [2]:
!pip install elsapy

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for elsapy from https://files.pythonhosted.org/packages/32/64/9926bbbc3c512c40aed0445675985fec037690e834531ece79e3792a257f/elsapy-0.5.1-py3-none-any.whl.metadata



[notice] A new release of pip is available: 23.2.1 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#query = 'subject-area(automotive)'

In [ ]:
import json
import time
from elsapy.elsclient import ElsClient
from elsapy.elssearch import ElsSearch

with open('config.json') as config_file:
    config = json.load(config_file)

api_key = config['apikey']  

client = ElsClient(api_key)

query = 'TITLE-ABS-KEY(automotive OR vehicle engineering) AND DOCTYPE(ar) AND NOT TITLE-ABS-KEY(review)'
results_per_page = 50 
total_results = 1000  

all_articles = [] 

def fetch_articles(query, start_index, results_per_page):
    search_query = f'{query}&start={start_index}&count={results_per_page}'
    search = ElsSearch(search_query, 'scopus')
    search.execute(client, get_all=False)
    
    articles = search.results
    
    for article in articles:
        article_data = {
            "title": article.get('dc:title', 'No title'),
            "authors": [author['authname'] for author in article.get('author', [])],
            "journal": article.get('prism:publicationName', 'No journal name'),
            "link": article.get('prism:url', 'No link'),
            "load-date": article.get('prism:coverDate', 'No load date'),
            "doi": article.get('prism:doi', 'No DOI')
        }
        all_articles.append(article_data)
    
    time.sleep(1)  

for start_index in range(0, total_results, results_per_page):
    fetch_articles(query, start_index, results_per_page)

with open('all_articles.json', 'w') as json_file:
    json.dump(all_articles, json_file, indent=4)

print("Finished.")